# Imports

In [1]:
import numpy as np
import scipy
from PIL import Image
import cv2 as cv
import torch
import torchvision.transforms as transforms
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import albumentations as A
from tqdm import tqdm
import json
import os
import random
import math
from collections import defaultdict

In [2]:
test = json.load(open('/Users/omar/Downloads/testing.json'))

In [3]:
test

{'info': {'description': 'Illegal landfills dataset',
  'version': '3.0',
  'year': 2024,
  'contributor': 'POLIMI - ARPA',
  'date_created': '2024/06/28'},
 'categories': [{'supercategory': 'Type_of_object',
   'id': 1,
   'name': 'Rubble/excavated earth and rocks'},
  {'supercategory': 'Type_of_object', 'id': 2, 'name': 'Bulky items'},
  {'supercategory': 'Type_of_object', 'id': 3, 'name': 'Fire Wood'},
  {'supercategory': 'Type_of_object', 'id': 4, 'name': 'Scrap'},
  {'supercategory': 'Type_of_object', 'id': 5, 'name': 'Plastic'},
  {'supercategory': 'Type_of_object', 'id': 6, 'name': 'Vehicles'},
  {'supercategory': 'Type_of_object', 'id': 7, 'name': 'Tires'},
  {'supercategory': 'Type_of_object', 'id': 8, 'name': 'Domestic appliances'},
  {'supercategory': 'Type_of_object', 'id': 9, 'name': 'Paper'},
  {'supercategory': 'Type_of_object',
   'id': 10,
   'name': 'Sludge-Zootechnical waste-Manure'},
  {'supercategory': 'Type_of_object', 'id': 11, 'name': 'Foundry waste'},
  {'super

In [4]:
segmented = test['annotations']

In [5]:
segmented

[{'id': 254,
  'image_id': 662,
  'segmentation': [[423.79211333333336,
    471.2642266666667,
    394.7662,
    482.1522133333333,
    403.1166733333333,
    497.43178,
    431.52664666666664,
    485.5651466666667,
    423.79211333333336,
    471.2642266666667]],
  'category_id': 17},
 {'id': 255,
  'image_id': 662,
  'segmentation': [[462.3606933333334,
    461.3521066666666,
    439.6012533333334,
    487.07443333333333,
    454.63181333333335,
    499.48621333333335,
    454.63181333333335,
    499.48621333333335,
    454.63181333333335,
    499.48621333333335,
    469.87149333333343,
    483.72496666666666,
    482.10882666666674,
    495.1974933333333,
    496.85511333333335,
    480.45097999999996,
    508.05730000000005,
    489.47194666666667,
    526.40778,
    475.02518,
    538.7869933333334,
    483.6498133333333,
    554.1071066666666,
    468.1872466666666,
    566.3232066666667,
    479.84880666666663,
    582.9423333333334,
    464.66571999999996,
    594.67996,
    4

In [6]:
len(segmented)

833

In [7]:
image_ids = {ann['image_id'] for ann in segmented}

In [8]:
len(image_ids)

166

## Get patches

In [9]:
images_dir = '/Users/omar/Downloads/images'
output_dir =  '/Users/omar/Downloads/segmentation_datasets'
patches_dir = os.path.join(output_dir, "patches"); os.makedirs(patches_dir, exist_ok=True)

os.makedirs(output_dir, exist_ok=True)
target = 6000
target_pos = target // 2
target_neg = target // 2
max_tries = 20000

In [10]:
# function from: https://github.com/nahitorres/AerialWaste/blob/main/compute_bbox_from_segmentation.ipynb
def compute_bbox(data):
    for annotation in data["annotations"]:
        seg = annotation["segmentation"][0]
        size = len(seg) - 1
        tuples = []
        w, h = [], []
        for i in range(0, size, 2):
            hcoord = seg[i+1]
            wcoord = seg[i]
            h.append(hcoord)
            w.append(wcoord)

        annotation["bbox"] =  [min(w), min(h), max(w)-min(w), max(h)-min(h)]

In [11]:
def create_patch_dataset(data, images_dir, output_dir, patch_size=256, target=target, max_tries=max_tries):
    patches_dir = os.path.join(output_dir, "patches"); os.makedirs(patches_dir, exist_ok=True)
    compute_bbox(data)
    id2file = {im['id']: im.get('file_name') for im in data.get('images', [])}
    anns_by_img = defaultdict(list)
    for ann in data.get('annotations', []):
        anns_by_img[ann['image_id']].append(ann)
    
    def rects_iou(a,b):
        ax1, ay1, ax2, ay2 = a[0], a[1], a[0]+a[2], a[1]+a[3]
        bx1, by1, bx2, by2 = b[0], b[1], b[0]+b[2], b[1]+b[3]
        ix1, iy1 = max(ax1, bx1), max(ay1, by1)
        ix2, iy2 = min(ax2, bx2), min(ay2, by2)
        iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
        inter = iw * ih
        union = a[2]*a[3] + b[2]*b[3] - inter
        return inter / union if union > 0 else 0
    
    def crop_from_center(img, cx, cy, size):
        h,w = img.shape[:2]; half = size//2
        x1 = int(max(0, min(w-size, cx - half))); y1 = int(max(0, min(h-size, cy - half)))
        return img[y1:y1+size, x1:x1+size], [x1, y1, size, size]
    
    def clip_coords_to_patch(segmentation, pbox, patch_size):
        """Clip coordinates to patch boundaries using simple arithmetic"""
        clipped_coords = []
        
        for i in range(0, len(segmentation), 2):
            # Get original coordinates
            x, y = segmentation[i], segmentation[i+1]
            
            # Shift to patch coordinates
            patch_x = x - pbox[0]
            patch_y = y - pbox[1]
            
            # Clip to patch boundaries
            patch_x = max(0, min(patch_size, patch_x))
            patch_y = max(0, min(patch_size, patch_y))
            
            clipped_coords.extend([patch_x, patch_y])
        
        return clipped_coords
    
    records = []
    pos_count, neg_count = 0, 0
    tries = 0

    while pos_count < target_pos and tries < max_tries:
        tries += 1
        ann = random.choice(data.get('annotations', []))
        img_id = ann['image_id']; fname = id2file.get(img_id)
        if not fname: continue

        img = cv.imread(os.path.join(images_dir, fname))
        if img is None: continue

        bx = ann['bbox']
        cx = int(bx[0] + random.uniform(0.3, 0.7)*bx[2] + random.uniform(-0.15, 0.15)*bx[2])
        cy = int(bx[1] + random.uniform(0.3, 0.7)*bx[3] + random.uniform(-0.15, 0.15)*bx[3])

        patch, pbox = crop_from_center(img, cx, cy, patch_size)
        if patch.shape[0] != patch_size or patch.shape[1] != patch_size:
            continue
        if rects_iou(pbox, bx) > 0:
            fname_out = f'patch_{pos_count+neg_count:06d}.png'
            cv.imwrite(os.path.join(patches_dir, fname_out), patch)
            
            # Use simple coordinate clipping
            clipped_coords = clip_coords_to_patch(ann.get('segmentation', [])[0], pbox, patch_size)
            
            records.append({
                'file_name': fname_out, 
                'segmentation': clipped_coords,  # Clipped coordinates
                'image_id': img_id, 
                'category_id': ann.get('category_id', 0),
                'label': 1
            })
            pos_count += 1
    
    tries = 0
    while neg_count < target_neg and tries < max_tries:
        tries += 1
        # Pick random image for negative sampling
        img_id = random.choice(list(anns_by_img.keys()))
        fname = id2file.get(img_id)
        if not fname: continue

        img = cv.imread(os.path.join(images_dir, fname))
        if img is None: continue

        h, w = img.shape[:2]
        if h < patch_size or w < patch_size:
            continue
        x1 = random.randint(0, w - patch_size); y1 = random.randint(0, h - patch_size)
        pbox = [x1, y1, patch_size, patch_size]
        overlap = False

        for ann in anns_by_img.get(img_id, []):
            if rects_iou(pbox, ann['bbox']) > 0.05:
                overlap = True
                break
        if overlap:
            continue
            
        patch = img[y1:y1+patch_size, x1:x1+patch_size]
        fname_out = f'patch_{pos_count+neg_count:06d}.png'
        cv.imwrite(os.path.join(patches_dir, fname_out), patch)
        
        records.append({
            'file_name': fname_out, 
            'segmentation': [], 
            'image_id': img_id, 
            'category_id': 0, 
            'label': 0
        })
        neg_count += 1
    
    out_json = os.path.join(output_dir, 'patches_labels.json')
    with open(out_json, 'w') as f:
        json.dump({'patches': records, 'counts': {'positive': pos_count, 'negative': neg_count}}, f, indent=2)
    
    return {'json': out_json, 'positive': pos_count, 'negative': neg_count}

In [12]:
create_patch_dataset(test, images_dir, output_dir)

{'json': '/Users/omar/Downloads/segmentation_datasets/patches_labels.json',
 'positive': 3000,
 'negative': 3000}